#### plot_siap_vs_adc_census_scatter.ipynb

**Author:** James Sayre
**Email:** jsayre@ucdavis.edu
**Date Modified:** 2026-06-26

**Description:** Generates panel (e) of Figure 1 in the paper: a hexbin
correlation plot of the DGSIAP municipal maize yield (2022) assigned to each
ADC (x) against the ADC-level maize yield from the 2022 Agricultural Census
(y). This is the "DGSIAP municipal average" downscaling benchmark
($R^2=0.49$, $N=94{,}898$, matching the DGSIAP row of the combined-season
accuracy table).

**Inputs:** adc_aef_hist_ens_eval.parquet (ADC-level census yield), siap_ag_prod_estimation_by_season.dta (DGSIAP municipal maize yield)
**Outputs:** siap_mun_vs_adc_census_yield.pdf (project plots/; copied to Overleaf by 04_copy_to_overleaf)

In [1]:
import os
import numpy as np
import pandas as pd

# -- Directories --------------------------------------------------
home_dir    =  os.path.expanduser("~")
proj_dir    =  os.path.join(home_dir, "Dropbox", "Projects", "Maize_prediction")
data_dir    =  os.path.join(proj_dir, "Data")
pred_dir    =  os.path.join(data_dir, "predictions")
plot_dir    =  os.path.join(proj_dir, "plots")
# 04_copy_to_overleaf is the single road to Overleaf; write to plots/ only

# -- Inputs -------------------------------------------------------
eval_path   =  os.path.join(pred_dir, "adc_aef_hist_ens_eval.parquet")  # ADC-level census yield + ens. preds
# 2026-08-28: switched from muni_grano_yields.csv (deprecated 2026-07-26 -- a
# DIFFERENT series from the one the models are scored on; see train/siap_yields.py)
# to the canonical DGSIAP by-season file, aggregated exactly like the DGSIAP
# benchmark rows of the accuracy tables (combined seasons, 2022).
siap_path   =  os.path.join(data_dir, "SIAP", "Cleaned", "siap_ag_prod_estimation_by_season.dta")  # DGSIAP municipal panel

# -- Outputs ------------------------------------------------------
out_pdf     =  os.path.join(plot_dir, "siap_mun_vs_adc_census_yield.pdf")  # Figure 1, panel (e)

In [2]:
# -- Build the DGSIAP-benchmark pairs at the ADC level --------------
ev      =  pd.read_parquet(eval_path)            # adc, muncode, yield (CA2022, combined season), ...
siap    =  pd.read_stata(siap_path)
siap["muncode"] =  siap["muncode"].apply(lambda x: str(int(x)).zfill(5))
s22     =  siap[(siap["name"] == "Maize") & (siap["year"] == 2022)]
s22     =  s22[~s22["muncode"].str.endswith("000")]
siap22  =  s22.groupby("muncode").agg(q=("q", "sum"), ha=("ha_planted", "sum")).reset_index()
siap22["yield_siap"] =  siap22["q"] / siap22["ha"]
siap22  =  siap22[["muncode", "yield_siap"]]

m  =  ev.merge(siap22, on="muncode", how="left").dropna(subset=["yield", "yield_siap"])
x  =  m["yield_siap"].values   # DGSIAP municipal yield, broadcast to each ADC
y  =  m["yield"].values        # ADC-level census (CA2022) yield

r2 =  1 - np.sum((y - x) ** 2) / np.sum((y - y.mean()) ** 2)
N  =  len(y)
print(f"N = {N:,}   R2 = {r2:.3f}")

N = 94,898   R2 = 0.492


In [3]:
# -- Matplotlib house style (pgf/LaTeX) --------------------------
from cycler import cycler
import matplotlib as mpl
mpl.use("pgf")
import matplotlib.pyplot as plt

base_font_size = 12
mpl.rcParams.update({
    "pgf.texsystem": "pdflatex", "pgf.rcfonts": False,
    "font.family": "serif", "font.serif": ["Times"], "axes.unicode_minus": False,
    "font.size": base_font_size, "axes.labelsize": base_font_size + 2,
    "xtick.labelsize": base_font_size, "ytick.labelsize": base_font_size,
    "axes.facecolor": "white", "figure.facecolor": "white",
    "axes.edgecolor": "#404040", "axes.labelcolor": "#404040", "xtick.color": "#404040", "ytick.color": "#404040",
    "grid.color": "#D0D0D0", "grid.linestyle": (0, (1, 3)), "grid.linewidth": 0.6,
    "pgf.preamble": "\\usepackage[T1]{fontenc}\n\\usepackage{mathptmx}\n",
})

# -- Hexbin scatter (viridis, matching Fig 1 panels a-d) ----------
hi  =  float(np.ceil(np.nanpercentile(np.concatenate([x, y]), 99)))
fig, ax =  plt.subplots(figsize=(4.6, 4.6))
ax.hexbin(x, y, gridsize=45, bins="log", cmap="viridis", mincnt=1, extent=(0, hi, 0, hi))
ax.plot([0, hi], [0, hi], ls="--", color="#D62728", lw=1.3, zorder=5)   # 1:1 line
ax.set_xlim(0, hi); ax.set_ylim(0, hi); ax.set_aspect("equal")
ax.set_xlabel(r"DGSIAP municipal yield (t/ha)")
ax.set_ylabel(r"ADC census yield (t/ha)")
ax.grid(True, alpha=0.35)
ax.text(0.04, 0.96, rf"$R^2 = {r2:.2f}$" + "\n" + rf"$N = {N:,}$".replace(",", r"{,}"),
        transform=ax.transAxes, va="top", ha="left", fontsize=base_font_size, color="white",
        bbox=dict(facecolor="black", alpha=0.45, edgecolor="none", boxstyle="round,pad=0.3"))
fig.tight_layout()

for o in (out_pdf, out_pdf):
    fig.savefig(o, bbox_inches="tight")
    print("wrote", o)

wrote /home/jdesktop/Dropbox/Projects/Maize_prediction/plots/siap_mun_vs_adc_census_yield.pdf


wrote /home/jdesktop/Dropbox/Projects/Maize_prediction/plots/siap_mun_vs_adc_census_yield.pdf
